# Lab 2 — De la matriz de modelación al scorecard · Financiera Andes
## Modelador de Riesgo de Crédito · Academia Bayes — **Entrega 2** (30% del curso)

**Fecha límite: domingo 13 de septiembre, 23:59 (hora de Chile).**

**Cómo se entrega:** por el canal del aula, el **link a este notebook en Colab** (con permiso de
lectura), **ejecutado de arriba a abajo sin errores** (`Entorno de ejecución → Reiniciar y
ejecutar todo`). Un notebook con celdas rojas se devuelve antes de corregirse.
**Atrasos:** −0,5 por día, máximo 3 días; después se considera no entregada. Si algo se complica,
avisa **antes** del plazo. **Feedback:** a más tardar el **miércoles 17 de septiembre**.

Este laboratorio continúa el Lab 1 y termina en el mismo punto donde cerramos la construcción
del modelo: la **tabla de puntos del scorecard**.

| Bloque | Qué construyes | Puntos |
|---|---|---|
| Selección | Estabilidad, poder, redundancia y VIF | 7 |
| Modelo | Stepwise, signos, significancia y Gini por muestra | 5 |
| Scorecard | Scaling PDO, puntos por tramo y score individual | 3 |

**Total: 15 puntos** → `nota = 1 + 9 × (puntaje / 15)`.

El notebook prepara la misma matriz `X` del Lab 1 con tu mismo correo. Esa preparación se entrega
hecha porque aquí no se vuelve a evaluar la fábrica de variables: el trabajo nuevo empieza en el
embudo multivariado.

> ⚠️ **Aviso sin spoiler: hay una variable plantada que tu embudo tiene que descartar.** Está en
> tu matriz, se ve inofensiva y **no debe llegar al modelo final**. No la vamos a nombrar acá: el
> punto del lab es que la caces con el procedimiento, no con el dato. Si te sobrevive hasta el
> scorecard, vuelve al **paso de estabilidad** (sección 1) y revísalo — ahí se cae.
> Descartarla y **justificar por escrito por qué** es parte de la nota.

**Invalidantes de la entrega** (los mismos de la pauta; llevan la nota a un **máximo de 4,0**):
variables con información posterior a t₀ · bins ajustados fuera de DEV · signos contraintuitivos
sin justificación escrita · métricas reportadas solo en DEV · usar una muestra, semilla o matriz
distinta de la preparada en la sección 0.

> La calibración de la PD, las bandas de riesgo y el punto de corte se trabajarán en el
> **Lab 3**. No se evalúan en este notebook.

---
## 0. Identificación y matriz `X` del Lab 1

⚠️ Usa el **mismo correo del Lab 1**. Tu muestra personal — y por lo
tanto todos tus números — dependen de él. Si cambias el correo, tus resultados
dejan de ser trazables con tu E1 y la corrección lo detecta.

In [ ]:
ID_ALUMNO = "tu.correo@ejemplo.cl"   # ← EDITA con tu correo de inscripción

import hashlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 140)
plt.rcParams["figure.figsize"] = (9, 4.2)
plt.rcParams["axes.grid"] = True; plt.rcParams["grid.alpha"] = 0.3

assert ID_ALUMNO != "tu.correo@ejemplo.cl", "⚠️ Edita ID_ALUMNO con tu correo real"
SEMILLA_PERSONAL = int(hashlib.sha256(f"ANDES-C1-2026-B|{ID_ALUMNO.strip().lower()}".encode()).hexdigest()[:8], 16) % 10**6
print(f"Tu semilla personal: {SEMILLA_PERSONAL}")

In [ ]:
# Misma versión fija de datos usada por el Lab 1 publicado.
BASE = "https://raw.githubusercontent.com/nexolabs-gh/datos-riesgo-credito/feaa1968d0808c44ff228506523b706b677e3020"
PREFIJO = "andes"

clientes       = pd.read_parquet(f"{BASE}/{PREFIJO}_clientes.parquet")
solicitudes    = pd.read_parquet(f"{BASE}/{PREFIJO}_solicitudes.parquet")
comportamiento = pd.read_parquet(f"{BASE}/{PREFIJO}_comportamiento.parquet")
bureau         = pd.read_parquet(f"{BASE}/{PREFIJO}_bureau.parquet")

# ---- Muestra personal (NO modificar esta celda) ----
# Ojo: el generador AVANZA con cada llamada — si re-ejecutas solo la línea de
# mis_clientes, la muestra cambia. Re-ejecuta esta celda completa (o «Reiniciar
# y ejecutar todo») y tu muestra personal será SIEMPRE la misma.
rng_p = np.random.default_rng(SEMILLA_PERSONAL)
ids = np.sort(clientes["id_cliente"].unique())
mis_clientes = set(rng_p.choice(ids, size=int(len(ids) * 0.60), replace=False))
clientes       = clientes[clientes["id_cliente"].isin(mis_clientes)].reset_index(drop=True)
solicitudes    = solicitudes[solicitudes["id_cliente"].isin(mis_clientes)].reset_index(drop=True)
comportamiento = comportamiento[comportamiento["id_cliente"].isin(mis_clientes)].reset_index(drop=True)
bureau         = bureau[bureau["id_cliente"].isin(mis_clientes)].reset_index(drop=True)
print(f"Misma muestra del Lab 1: {len(clientes):,} clientes · {len(solicitudes):,} solicitudes")

### Recordatorio: el diccionario de datos, a mano

Seguimos con las mismas cuatro tablas de Financiera Andes, así que dejamos el
diccionario cargado otra vez. Cada variable que fabriquemos hoy nace de una
columna de estas tablas: si no sabes qué mide la columna, no puedes defender la
variable frente a un comité.

[`diccionario_andes.md`](https://github.com/nexolabs-gh/datos-riesgo-credito/blob/main/diccionario_andes.md) ·
la celda siguiente lo muestra completo y deja `mostrar_diccionario(...)` definida.

In [ ]:
import urllib.request

from IPython.display import Markdown, display

URL_DICCIONARIO = f"{BASE}/diccionario_{PREFIJO}.md"
_DICCIONARIO = urllib.request.urlopen(URL_DICCIONARIO).read().decode("utf-8")


def mostrar_diccionario(tabla=None):
    """Muestra el diccionario de datos oficial; sin argumento, lo muestra completo."""
    if tabla is None:
        display(Markdown(_DICCIONARIO))
        return
    secciones = _DICCIONARIO.split("\n## ")[1:]
    elegidas = [s for s in secciones if s.lower().startswith(tabla.lower())]
    if not elegidas:
        disponibles = [s.split("(")[0].strip() for s in secciones]
        raise ValueError(f"No existe la tabla {tabla!r}. Disponibles: {disponibles}")
    display(Markdown("## " + elegidas[0]))


mostrar_diccionario()

### La misma matriz que alimenta el Lab 2

Las dos celdas siguientes reconstruyen la matriz `X` trabajada en el Lab 1. Mantienen la
misma versión de los datos, semilla personal, población, target 90+ a 12 meses, cosechas,
ventanas y variables de Financiera Andes.

Estas celdas son el **insumo entregado** del Lab 2. No las modifiques. El objetivo de este
laboratorio comienza después, con la selección multivariada.

In [ ]:
# ---- Insumo del Lab 1: target, población y muestras ----
mora_w = comportamiento.pivot_table(index="id_cliente", columns="mes",
                                    values="dias_mora", aggfunc="first")
meses = list(mora_w.columns)
pos_mes = {m: k for k, m in enumerate(meses)}
M = mora_w.to_numpy()
fila_de = {c: k for k, c in enumerate(mora_w.index)}

cursadas = solicitudes[solicitudes["aprobada"]].copy()
cursadas["_t"] = cursadas["fecha_solicitud"].map(pos_mes)
cursadas["_i"] = cursadas["id_cliente"].map(fila_de)
peor = np.full(len(cursadas), np.nan)
for k, (i, t0) in enumerate(zip(cursadas["_i"], cursadas["_t"])):
    if t0 + 12 <= len(meses) - 1:
        peor[k] = M[i, t0 + 1: t0 + 13].max()
cursadas["peor_dpd_12m"] = peor
cursadas["malo"] = np.where(np.isnan(peor), np.nan, (peor >= 90).astype(float))
cursadas["indeterminado"] = (peor >= 30) & (peor < 90)

antig = clientes.set_index("id_cliente")["fecha_alta_cliente"].map(
    lambda s: (pd.Period(s, freq="M") - pd.Period(meses[0], freq="M")).n)
periodo = cursadas[cursadas["fecha_solicitud"].between("2024-09", "2025-06")]
periodo = periodo.sort_values("fecha_solicitud", kind="stable").drop_duplicates(
    "id_cliente", keep="first").copy()
periodo["antiguedad_meses"] = periodo["_t"] - periodo["id_cliente"].map(antig)
base = periodo[periodo["malo"].notna() & ~periodo["marca_fraude"]
               & (periodo["antiguedad_meses"] >= 6)].copy()
rng = np.random.default_rng(SEMILLA_PERSONAL)
base["muestra"] = np.where(base["fecha_solicitud"] <= "2025-01",
                           np.where(rng.random(len(base)) < 0.75, "DEV", "HO"), "OOT")
ttd_pob = cursadas[cursadas["fecha_solicitud"] >= "2025-07"].copy()
ttd_pob["antiguedad_meses"] = ttd_pob["_t"] - ttd_pob["id_cliente"].map(antig)
ttd_pob = ttd_pob[ttd_pob["antiguedad_meses"] >= 6]
ttd_pob["muestra"] = "TTD"
poblacion = pd.concat([base, ttd_pob], ignore_index=True)
print(poblacion["muestra"].value_counts().reindex(["DEV", "HO", "OOT", "TTD"]).to_string())

In [ ]:
# ---- Insumo del Lab 1: panel, fábrica, fotos y ratios ----
def pivotar(df, columna):
    ancho = df.pivot_table(index="id_cliente", columns="mes", values=columna, aggfunc="first")
    return ancho.reindex(index=mora_w.index, columns=meses).to_numpy()

PANEL = {c: pivotar(comportamiento, c) for c in
         ["dias_mora", "saldo_linea", "cupo_linea", "saldo_tc", "cupo_tc", "saldo_consumo",
          "monto_pagado", "monto_facturado", "abonos_cuenta", "saldo_ahorro", "n_productos"]}
PANEL.update({c: pivotar(bureau, c) for c in
              ["deuda_otras_inst", "n_otras_inst", "consultas_mes", "peor_mora_sistema"]})
with np.errstate(divide="ignore", invalid="ignore"):
    PANEL["uso_linea"] = np.where(PANEL["cupo_linea"] > 0,
                                  PANEL["saldo_linea"] / PANEL["cupo_linea"], np.nan)
    PANEL["uso_tc"] = np.where(PANEL["cupo_tc"] > 0,
                               PANEL["saldo_tc"] / PANEL["cupo_tc"], np.nan)
PANEL["deuda_interna"] = PANEL["saldo_linea"] + PANEL["saldo_tc"] + PANEL["saldo_consumo"]
PANEL["deuda_total"] = PANEL["deuda_interna"] + PANEL["deuda_otras_inst"]
PANEL["en_mora"] = np.where(np.isnan(PANEL["dias_mora"]), np.nan,
                            (PANEL["dias_mora"] > 0).astype(float))
PANEL["en_mora_sistema"] = np.where(np.isnan(PANEL["peor_mora_sistema"]), np.nan,
                                    (PANEL["peor_mora_sistema"] > 0).astype(float))

def prom(V):  return np.nanmean(V, axis=1)
def vmax(V):  return np.nanmax(V, axis=1)
def suma(V):  return np.nansum(V, axis=1)
def delta(V): return V[:, -1] / np.maximum(V[:, 0], 1) - 1
def recencia(V):
    evento = np.nan_to_num(V, nan=0.0) > 0
    k = evento.shape[1]
    ultimo = k - 1 - np.argmax(evento[:, ::-1], axis=1)
    return np.where(evento.any(axis=1), k - ultimo, k + 1)
AGREGADORES = {"prom": prom, "max": vmax, "suma": suma, "delta": delta, "recencia": recencia}

def agregar_ventana(matriz, filas, t0s, k, funcion):
    salida = np.full(len(filas), np.nan)
    for t0 in np.unique(t0s):
        sel = t0s == t0
        salida[sel] = funcion(matriz[filas[sel], max(t0 - k, 0):t0])   # t0 NO entra
    return salida

FABRICA = [
    ("dias_mora", "max", (3, 6, 12), "dias_mora_max_{k}m"),
    ("dias_mora", "prom", (3, 6, 12), "dias_mora_prom_{k}m"),
    ("en_mora", "suma", (3, 6, 12), "n_meses_mora_{k}m"),
    ("en_mora", "recencia", (12,), "meses_desde_mora_12m"),
    ("uso_linea", "prom", (3, 6, 12), "uso_linea_prom_{k}m"),
    ("uso_linea", "max", (3, 6, 12), "uso_linea_max_{k}m"),
    ("uso_tc", "prom", (3, 6, 12), "uso_tc_prom_{k}m"),
    ("uso_tc", "max", (3, 6, 12), "uso_tc_max_{k}m"),
    ("saldo_consumo", "prom", (3, 6, 12), "saldo_consumo_prom_{k}m"),
    ("deuda_interna", "prom", (3, 6, 12), "deuda_interna_prom_{k}m"),
    ("deuda_total", "prom", (3, 6, 12), "deuda_total_prom_{k}m"),
    ("deuda_total", "delta", (6, 12), "deuda_total_delta_{k}m"),
    ("monto_pagado", "suma", (3, 6, 12), "pagos_{k}m"),
    ("monto_facturado", "suma", (3, 6, 12), "facturacion_{k}m"),
    ("abonos_cuenta", "prom", (3, 6, 12), "abonos_prom_{k}m"),
    ("saldo_ahorro", "prom", (3, 6, 12), "saldo_ahorro_prom_{k}m"),
    ("deuda_otras_inst", "prom", (3, 6, 12), "deuda_otras_prom_{k}m"),
    ("deuda_otras_inst", "delta", (6,), "delta_deuda_otras_{k}m"),
    ("n_otras_inst", "prom", (3, 6, 12), "n_otras_inst_prom_{k}m"),
    ("peor_mora_sistema", "max", (3, 6, 12), "peor_mora_sistema_{k}m"),
    ("consultas_mes", "suma", (3, 6, 12), "consultas_{k}m"),
    ("consultas_mes", "recencia", (12,), "meses_desde_consulta_12m"),
]
META = ["id_solicitud", "id_cliente", "fecha_solicitud", "_t", "_i",
        "muestra", "malo", "indeterminado", "peor_dpd_12m"]
filas_p, t0s_p = poblacion["_i"].to_numpy(), poblacion["_t"].to_numpy()
X = poblacion[META + ["antiguedad_meses", "monto_solicitado", "plazo_meses",
                      "destino", "canal"]].copy()
for serie, agg, ventanas, plantilla in FABRICA:
    for k in ventanas:
        X[plantilla.format(k=k)] = agregar_ventana(PANEL[serie], filas_p, t0s_p, k,
                                                   AGREGADORES[agg])
t_ult = np.maximum(t0s_p - 1, 0)
for serie in ["dias_mora", "saldo_linea", "cupo_linea", "saldo_tc", "cupo_tc",
              "saldo_consumo", "saldo_ahorro", "deuda_otras_inst", "n_productos",
              "peor_mora_sistema", "uso_linea", "uso_tc"]:
    X[f"{serie}_ult"] = PANEL[serie][filas_p, t_ult]
ingreso = X["abonos_prom_6m"].to_numpy()
deuda_ult = (X["saldo_linea_ult"] + X["saldo_tc_ult"]
             + X["saldo_consumo_ult"] + X["deuda_otras_inst_ult"])
X["carga_financiera"] = deuda_ult / np.maximum(ingreso, 1)
X["pago_sobre_fact_6m"] = X["pagos_6m"] / np.maximum(X["facturacion_6m"], 1)
X["pago_sobre_fact_12m"] = X["pagos_12m"] / np.maximum(X["facturacion_12m"], 1)
X["deuda_int_sobre_sistema"] = (X["saldo_linea_ult"] + X["saldo_tc_ult"]
                                + X["saldo_consumo_ult"]) / np.maximum(deuda_ult, 1)
X["ahorro_sobre_ingreso"] = X["saldo_ahorro_ult"] / np.maximum(ingreso, 1)
X["monto_sobre_ingreso"] = X["monto_solicitado"] / np.maximum(ingreso, 1)
X["cuota_est_sobre_ingreso"] = (X["monto_solicitado"] / X["plazo_meses"]) / np.maximum(ingreso, 1)
X = X.merge(clientes[["id_cliente", "edad", "sexo", "region", "estado_civil",
                      "nivel_educacional", "tipo_empleo", "renta_liquida",
                      "n_dependientes"]], on="id_cliente", how="left")
X["renta_vs_abonos"] = X["renta_liquida"] / np.maximum(ingreso, 1)

predictoras = [c for c in X.columns if c not in META]
filtro = X["malo"].notna() & ~X["indeterminado"]
dev = X[(X["muestra"] == "DEV") & filtro]
ho  = X[(X["muestra"] == "HO") & filtro]
oot = X[(X["muestra"] == "OOT") & filtro]
ttd = X[X["muestra"] == "TTD"]
print(f"TU MATRIZ: {len(X):,} filas × {len(predictoras)} candidatas")
print(f"DEV={len(dev):,} ({dev['malo'].mean():.1%} malos) · HO={len(ho):,} · "
      f"OOT={len(oot):,} · TTD={len(ttd):,}")

### Las herramientas del curso (se entregan hechas)

`binear()` y `tabla_woe()` de la clase 2, con el argumento `ref` que vimos en la
clase 3: los cortes se calculan en la muestra de referencia (DEV) y se **aplican**
a cualquier otra. Léelas — las vas a usar en todo el lab.

In [ ]:
def binear(x, bins=5, umbral_moda=0.35, ref=None):
    """Binning del curso. Con `ref`, los cortes se calculan en `ref` y se aplican a `x`."""
    x = pd.Series(x).reset_index(drop=True)
    base_b = x if ref is None else pd.Series(ref).reset_index(drop=True)
    if not pd.api.types.is_numeric_dtype(base_b):
        return x.astype(str).where(x.notna(), "MISSING"), None
    if base_b.nunique(dropna=True) <= bins:
        return x.astype(str).where(x.notna(), "MISSING"), None
    moda = base_b.mode().iloc[0]
    frac_moda = (base_b == moda).mean()
    resto = base_b[base_b != moda] if frac_moda > umbral_moda else base_b
    cortes = np.unique(np.nanquantile(resto.dropna(), np.linspace(0, 1, bins + 1)))
    if len(cortes) < 3:
        return x.astype(str).where(x.notna(), "MISSING"), None
    cortes[0], cortes[-1] = -np.inf, np.inf
    categorias = pd.cut(x, cortes)
    orden = [str(c) for c in categorias.cat.categories]
    etiquetas = categorias.astype(str)
    if frac_moda > umbral_moda:
        etiquetas = etiquetas.where(x != moda, f"= {moda:.4g}")
        orden = [f"= {moda:.4g}"] + orden
    return etiquetas.where(x.notna(), "MISSING"), orden + ["MISSING"]


def tabla_woe(x, y, bins=5, ref=None):
    """Tabla WoE de una variable y su IV (convención: target 1 = malo)."""
    etiquetas, orden = binear(x, bins, ref=ref)
    tab = pd.DataFrame({"bin": etiquetas.values, "y": pd.Series(y).values}) \
            .groupby("bin")["y"].agg(["count", "sum"])
    tab.columns = ["n", "malos"]
    if orden:
        tab = tab.reindex([o for o in orden if o in tab.index])
    tab["buenos"] = tab["n"] - tab["malos"]
    tab["tasa_malos"] = tab["malos"] / tab["n"]
    p_malos  = (tab["malos"] + 0.5) / (tab["malos"].sum() + 0.5 * len(tab))
    p_buenos = (tab["buenos"] + 0.5) / (tab["buenos"].sum() + 0.5 * len(tab))
    tab["woe"] = np.log(p_buenos / p_malos)
    tab["iv_aporte"] = (p_buenos - p_malos) * tab["woe"]
    return tab, float(tab["iv_aporte"].sum())


ivs = pd.Series({v: tabla_woe(dev[v], dev["malo"])[1] for v in predictoras}
                ).sort_values(ascending=False)
print(f"IV calculado (en DEV) para {len(ivs)} candidatas")

---
# Modelo e interpretación
### Lab 2 · 15 puntos

Desde la matriz `X` del Lab 1 hasta una scorecard utilizable: estabilidad, redundancia,
stepwise, validación inicial y puntos por atributo.

---
## 1. Estabilidad primero: PSI contra OOT y contra TTD (3 puntos)

**(1a)** Implementa `psi()`. Recuerda la clase 3:

$$\text{PSI} = \sum_b \left(p_b^{\,\text{actual}} - p_b^{\,\text{esperado}}\right)\,
\ln\!\frac{p_b^{\,\text{actual}}}{p_b^{\,\text{esperado}}}$$

- Los bins se definen con **deciles de la muestra esperada** (DEV) y se aplican a
  la actual (para eso está `binear(..., ref=...)`, o cuantiles directos).
- Las categóricas se comparan nivel a nivel.
- `MISSING` es un bin más.
- Suma una constante pequeña (p. ej. `1e-4`) a las proporciones para evitar
  logaritmos de cero.

**(1b)** Calcula el PSI de **todas** tus candidatas contra OOT y contra TTD, y
descarta las que superen **0,25 contra TTD**. Muestra las 12 más inestables y el
conteo del embudo.

⚠️ Vas a encontrar un tercer grupo: variables cuyo PSI da `NaN` (los deciles de
DEV colapsan por la masa de ceros — pasa con las moras cortas). Regla del curso,
conservadora: **sin PSI medible no hay certificado de estabilidad → tampoco
siguen**. Repórtalas aparte, con su IV: un descarte silencioso es indefendible.

In [ ]:
def psi(esperado, actual, bins=10):
    # 👉 TU CÓDIGO:
    #  1) si `esperado` es categórica: niveles como bins (MISSING incluido)
    #  2) si es numérica: cortes por deciles de `esperado`, aplicados a ambas
    #  3) proporciones por bin (+1e-4) y la suma de (pa - pe) * ln(pa / pe)
    ...


# 👉 TU CÓDIGO (1b): tabla `estabilidad` con columnas iv, psi_oot, psi_ttd
# para TODAS tus candidatas, y las tres listas del embudo de estabilidad:
#   FUERA_PSI   (psi_ttd > 0.25)          NO_MEDIBLES (psi_ttd es NaN)
#   ESTABLES    (psi_ttd <= 0.25)
estabilidad = ...
FUERA_PSI = ...
NO_MEDIBLES = ...
ESTABLES = ...


In [ ]:
# ---- Verificación automática (NO modificar): el PSI y el drift plantado ----
if psi(dev["uso_linea_prom_12m"], dev["uso_linea_prom_12m"]) is None or estabilidad is ...:
    print("⏭️ Ítem 1 aún sin resolver — completa psi() y el embudo de estabilidad antes de entregar")
else:
    assert psi(dev["uso_linea_prom_12m"], dev["uso_linea_prom_12m"]) < 0.01, \
        "❌ El PSI de una muestra contra sí misma debe ser ≈ 0"
    _candidatas_cat = [v for v in predictoras
                       if not pd.api.types.is_numeric_dtype(X[v])]
    _p_cat = psi(dev[_candidatas_cat[0]], ttd[_candidatas_cat[0]])
    assert _p_cat == _p_cat, "❌ Tu psi() no maneja variables categóricas"
    assert len(FUERA_PSI) >= 6, (
        f"❌ Solo {len(FUERA_PSI)} variables con PSI_TTD > 0,25: en Andes hay drift "
        "real plantado y tu PSI no lo está viendo. Revisa los bins de referencia.")
    _categoricas_fuera = [v for v in FUERA_PSI
                          if not pd.api.types.is_numeric_dtype(X[v])]
    assert _categoricas_fuera, (
        "❌ La variable inestable plantada de Andes es categórica y no está en tu "
        "lista de descartes: revisa que tu psi() compare niveles, no solo números.")
    print(f"✅ PSI OK · {len(FUERA_PSI)} variables descartadas por inestabilidad, "
          f"entre ellas {_categoricas_fuera}")

**✍️ Tu justificación (1c):** identifica la variable categórica que tu PSI
marca como inestable. Explica: (i) qué cambio del negocio podría producir ese
drift, (ii) por qué una variable así puede ser peligrosa aunque tenga señal, y
(iii) por qué conviene contrastar DEV con TTD además de mirar OOT.



---
## 2. Poder mínimo: IV ≥ 0,10 (0,5 punto)

De las estables, quédate con las de IV ≥ 0,10 (calculado **solo en DEV**, como
siempre). Documenta el conteo del embudo.

In [ ]:
# 👉 TU CÓDIGO: DataFrame `cand` (estables con IV >= 0.10) ORDENADO por IV
# descendente, y el print del embudo hasta aquí
cand = ...


---
## 3. Redundancia: correlación entre WoE (2,5 puntos)

**(3a)** Transforma tus candidatas al **WoE de DEV** (los mapas se entregan
hechos) y calcula la matriz de correlación absoluta. ¿Cuántos pares superan 0,90?

**(3b)** Aplica el **descarte greedy** del curso: recorre las candidatas de mayor
a menor IV; cada una entra a la selección solo si su correlación con **todas**
las ya seleccionadas es ≤ 0,70.

In [ ]:
# Mapas bin -> WoE ajustados en DEV (se entregan hechos; los usarás en todo el lab)
def a_woe(df, variables):
    """Transforma `df` al WoE de DEV: binea con cortes de DEV y mapea su WoE."""
    F = pd.DataFrame(index=range(len(df)))
    for v in variables:
        etiquetas, _ = binear(df[v], ref=dev[v])
        F[v] = etiquetas.map(MAPAS[v]).fillna(0.0).values
    return F

if cand is ...:
    MAPAS, W = {}, None
    print("⏭️ Falta la tabla `cand` del ítem 2 — esta celda se completa sola cuando la tengas")
else:
    MAPAS = {v: tabla_woe(dev[v], dev["malo"])[0]["woe"] for v in cand.index}
    W = a_woe(dev, cand.index)
    print(f"frame WoE: {W.shape[0]:,} filas × {W.shape[1]} candidatas")

In [ ]:
# 👉 TU CÓDIGO (3a): matriz de correlación absoluta entre los WoE y conteo
# de pares > 0.90 (muestra los 10 mayores)
corr = ...

# 👉 TU CÓDIGO (3b): descarte greedy por IV con umbral 0.70
# (recorre cand.index, que ya está ordenado por IV descendente)
seleccion = ...


In [ ]:
# ---- Verificación automática (NO modificar): la shortlist ----
if seleccion is ...:
    print("⏭️ Ítem 3 aún sin resolver — completa el descarte greedy antes de entregar")
else:
    assert 15 <= len(seleccion) <= 35, (
        f"❌ Tu shortlist tiene {len(seleccion)} variables (esperado 15–35). "
        "Umbral mal aplicado o correlación calculada sobre los valores crudos.")
    _max_corr = max(corr.loc[a, b] for i, a in enumerate(seleccion)
                    for b in seleccion[i + 1:])
    assert _max_corr <= 0.70 + 1e-9, \
        f"❌ Quedan pares sobre 0,70 en tu selección (máximo: {_max_corr:.3f})"
    assert not (set(FUERA_PSI) & set(seleccion)), \
        "❌ Hay variables inestables (PSI) dentro de tu selección"
    print(f"✅ Shortlist OK · {len(seleccion)} variables · correlación máxima {_max_corr:.3f}")

**✍️ Tu justificación (3c):** agrupa tu shortlist por **concepto de negocio**
(mora propia, utilización, deuda interna, deuda externa, pagos, ahorro/ingreso,
consultas, demografía/relación). ¿Quedaron todos los conceptos representados?
¿Hay alguno sobre-representado? ¿Echas de menos alguna variable que el embudo
descartó y que tú habrías conservado por negocio — y qué harías al respecto?



---
## 4. Colinealidad múltiple: VIF (1 punto)

Calcula el VIF de tu selección (sobre los WoE estandarizados, con constante).
Reporta el máximo y decide si descarta algo. Recuerda: **se reporta aunque no
encuentre nada** — ausencia de evidencia no es evidencia de ausencia.

In [ ]:
# 👉 TU CÓDIGO: VIF por variable de tu selección, ordenado descendente
from statsmodels.stats.outliers_influence import variance_inflation_factor
import statsmodels.api as sm

vifs = ...


---
## 5. Stepwise sobre WoE: el modelo (3 puntos)

**(5a)** Implementa el stepwise *forward con revisión* de la clase 3:

1. Parte sin variables; en cada ronda prueba todas las restantes y **entra** la
   que más mejora la log-verosimilitud, si su p-valor < 0,05.
2. Tras cada entrada, revisa las ya incluidas: si alguna perdió significancia,
   sale **la peor** — y se re-estima antes de volver a mirar (sacar varias de
   golpe usa p-valores obsoletos).
3. Detente cuando ninguna candidata aporte (o al llegar a 14 variables — el tope
   de parsimonia del curso).

**(5b)** Ajusta el modelo final, **imprime la tabla de coeficientes y p-valores**
(se evalúa) y **verifica los signos**: con nuestra convención (WoE alto = bin
bueno) todos los coeficientes deben ser **negativos**.

> ⚠️ Con algunas muestras personales el stepwise deja entrar una variable con
> coeficiente **positivo** (la verificación de abajo te lo dirá). Es la política
> de signos de la clase en acción: sácala de tu `seleccion`
> (`seleccion.remove("la_variable")`), re-corre esta celda y **declara la
> exclusión en tu justificación 5c**. No es un error tuyo: es la decisión que
> un modelador toma todas las semanas.

In [ ]:
y_dev = dev["malo"].values

def ajustar(columnas):
    Xm = sm.add_constant(W[columnas], has_constant="add")
    return sm.Logit(y_dev, Xm).fit(disp=0, maxiter=200)

elegidas = []
# 👉 TU CÓDIGO — stepwise forward con revisión backward. Esqueleto sugerido:
#    restantes = list(seleccion)
#    while restantes:
#        ...
#  1) prueba cada candidata de `restantes` sobre `elegidas + [v]`
#  2) quédate con la de mayor log-verosimilitud (m.llf) cuyo p-valor
#     (m.pvalues[v]) sea < 0.05; si ninguna cumple, corta el while
#  3) agrégala a `elegidas`, imprímela con su p-valor
#  4) revisión backward: MIENTRAS haya incluidas con p >= 0.05, saca LA PEOR
#     (mayor p) y re-ajusta antes de volver a revisar
#  5) corta al llegar a 14 variables
#  ⚠️ El cuerpo del while es tuyo: no dejes un while que no avance.
...

if elegidas:
    modelo = ajustar(elegidas)
    print(f"EMBUDO: {len(seleccion)} → {len(elegidas)} variables finales")

In [ ]:
# ---- Verificación automática (NO modificar): parsimonia y signos ----
if not elegidas:
    print("⏭️ Ítem 5 aún sin resolver — implementa el stepwise antes de entregar")
else:
    assert 5 <= len(elegidas) <= 14, (
        f"❌ {len(elegidas)} variables finales (esperado 5–14). Con menos de 5 el "
        "embudo quedó roto: revisa el criterio de entrada del stepwise.")
    if len(elegidas) < 8:
        print(f"ℹ️ {len(elegidas)} variables finales: menos que las 8 de la corrida docente. "
              "Es legítimo si tu stepwise agotó las candidatas significativas — "
              "decláralo en 5c.")
    _positivos = [v for v in elegidas if modelo.params[v] > 0]
    assert not _positivos, (
        f"❌ Coeficientes POSITIVOS: {_positivos}. Con WoE alto = bin bueno, todos "
        "deben ser negativos. Pasa con algunas muestras personales: saca esa variable "
        "de tu `seleccion` (seleccion.remove(...)), re-corre desde el stepwise y "
        "declara la exclusión en 5c (es la política de signos de la clase).")
    _no_sig = [v for v in elegidas if modelo.pvalues[v] >= 0.05]
    assert not _no_sig, f"❌ Variables no significativas en el modelo final: {_no_sig}"
    print(f"✅ Modelo OK · {len(elegidas)} variables · todos los signos negativos · "
          "todas significativas")

**✍️ Tu justificación (5c):** el requisito de la entrega — **una línea por
variable final** con el concepto de riesgo que aporta al modelo (como la tabla de
la clase 3 para el Banco Austral). Si dos variables comparten concepto (p. ej.
utilización a 3 y 12 meses), explica por qué ambas se ganan su lugar (pista:
nivel vs tendencia reciente).



---
## 6. ¿Funciona? Gini por muestra y estabilidad de coeficientes (2 puntos)

**(6a)** Calcula el Gini de tu modelo en DEV, HO y OOT (recuerda: HO y OOT se
transforman con `a_woe`, que aplica los cortes y WoE **de DEV**). Lee las caídas
contra las reglas prácticas de la clase (DEV→HO < 0,10 · DEV→OOT < 0,15).

**(6b)** Re-ajusta el modelo en HO con las mismas variables y compara los
coeficientes. ¿Alguno se invierte? ¿Con qué significancia?

In [ ]:
from sklearn.metrics import roc_auc_score

# 👉 TU CÓDIGO (6a): gini(df) = 2*AUC - 1 usando modelo.predict sobre a_woe(df, elegidas)
# y el reporte para DEV / HO / OOT con las dos caídas
def gini(df):
    ...

# 👉 TU CÓDIGO (6b): re-ajusta en HO (mismas variables) y compara coeficientes


**✍️ Tu justificación (6c):** la **lectura honesta** que exige la entrega.
(i) ¿Tus caídas de Gini están dentro de las reglas prácticas? ¿Qué concluyes?
(ii) Si algún coeficiente cambió de signo en HO: ¿es una inversión significativa
o una estimación indistinguible de cero? ¿Qué se reporta en cada caso?
(iii) ¿Por qué HO sirve para *contrastar* pero no para *re-estimar*?



---
## 7. El scorecard: scaling PDO (3 puntos)

Convención del curso (la misma de la clase): **PDO = 20, score base 600 a odds
50:1**. Calcula `factor` y `offset`, construye la **tabla de puntos por bin** y
reporta el rango de puntos por variable y la distribución del score en DEV.

$$\text{puntos}(v, b) = -\left(\beta_v\,\text{WoE}_{v,b} + \frac{\beta_0}{n}\right)
\times \text{factor} + \frac{\text{offset}}{n}$$

In [ ]:
PDO, SCORE_BASE, ODDS_BASE = 20, 600, 50

# 👉 TU CÓDIGO: factor y offset según las fórmulas de la clase
factor = ...
offset = ...

# 👉 TU CÓDIGO: DataFrame `scorecard` con (variable, bin, puntos) usando MAPAS
# y el modelo final; muestra el rango de puntos por variable, ordenado
scorecard = ...

# 👉 TU CÓDIGO: score_de(df) -> (score, pd) y la distribución del score en DEV
def score_de(df):
    ...


In [ ]:
# ---- Verificación automática (NO modificar): el scaling ----
if factor is ... or offset is ...:
    print("⏭️ Ítem 7 aún sin resolver — calcula factor y offset antes de entregar")
else:
    assert abs(factor - 28.8539) < 0.01, f"❌ factor = {factor:.3f} (esperado 28.854)"
    assert abs(offset - 487.1229) < 0.01, f"❌ offset = {offset:.2f} (esperado 487.12)"
    _chk = score_de(dev)
    if _chk is None:
        print("⏭️ Falta score_de() del ítem 7 — complétala antes de entregar")
    else:
        _score_chk, _ = _chk
        assert np.isfinite(_score_chk).all(), "❌ Hay scores no finitos: revisa la protección del log"
        assert 450 < np.median(_score_chk) < 700, \
            f"❌ Mediana del score fuera de rango razonable: {np.median(_score_chk):.0f}"
        print(f"✅ Scaling OK · factor {factor:.3f} · offset {offset:.2f} · "
              f"mediana DEV {np.median(_score_chk):.0f}")

**✍️ Tu justificación (7b):** elige dos variables de tu scorecard y explica
qué ocurre con sus puntos a medida que cambia el riesgo. Después compara sus
rangos de puntos: ¿cuál puede mover más el score de un cliente y por qué el rango
no depende solo del coeficiente? Cierra explicando qué representa el score total
de una solicitud.



---
## Cierre y entrega

**Checklist antes de entregar:**
- [ ] `Reiniciar y ejecutar todo` corre sin errores y aparecen las verificaciones
- [ ] Usaste el mismo correo del Lab 1
- [ ] Completaste las justificaciones 1c, 3c, 5c, 6c y 7b
- [ ] Los bins y WoE se estimaron en DEV y se aplicaron sin reajustar a HO y OOT
- [ ] Reportaste Gini en DEV, HO y OOT
- [ ] La scorecard contiene una fila por variable y tramo, con sus puntos
- [ ] No incluiste calibración, bandas ni punto de corte en esta entrega

**Cómo se corrige:** 15 puntos, con `nota = 1 + 9 × (puntaje / 15)`.

> Guarda este notebook. El modelo y la scorecard serán el punto de partida del
> **Lab 3**, donde trabajaremos calibración, bandas de riesgo y estrategia.